# J2 après-midi · 3. Limiter le débit, file d'attente

Livret de TP J2 après-midi, **exercices 6, 7, 8 et 11**.

In [1]:
import os
import redis

# Le kit donne l'adresse de Redis dans la variable REDIS_URL : redis://redis:6379/0
r = redis.from_url(os.environ["REDIS_URL"], decode_responses=True)
r.ping()

import json
import time

## Exercice 6 : Limiter le nombre de requêtes

NYC Eats accepte au plus 5 recherches par visiteur et par minute. Méthode : un compteur
`limite:<pseudo>` qu'on augmente à chaque requête ; à la première requête, on lui donne une
durée de vie de 60 secondes. Tant que le compteur ne dépasse pas 5, la requête passe.

**Question 1.** Écrivez `autoriser(pseudo, limite=5, fenetre=60)`, qui renvoie `True` si la
requête passe, `False` sinon.

In [2]:
def autoriser(pseudo, limite=5, fenetre=60):
    cle = f"limite:{pseudo}"
    # À COMPLÉTER : augmenter le compteur ; à la première requête, lui donner une durée de vie
    nb_requetes = r.incr(cle)
    if nb_requetes == 1:
        r.expire(cle, fenetre)
    return nb_requetes <= limite

**Question 2.** Léa lance 8 recherches d'affilée, puis Karim une seule. Qui passe ?

In [3]:
r.delete("limite:lea", "limite:karim")   # on repart de zéro
for numero in range(1, 9):
    print("Léa, recherche", numero, ":", autoriser("lea"))
print("Karim, recherche 1 :", autoriser("karim"))

Léa, recherche 1 : True
Léa, recherche 2 : True
Léa, recherche 3 : True
Léa, recherche 4 : True
Léa, recherche 5 : True
Léa, recherche 6 : False
Léa, recherche 7 : False
Léa, recherche 8 : False
Karim, recherche 1 : True


**Question 3.** Que valent le compteur de Léa et son TTL ?

In [4]:
print(r.get("limite:lea"), r.ttl("limite:lea"))

8 60


**Question 4.** Avec une fenêtre de 3 secondes et une limite de 2, vérifiez que le compteur
repart à zéro quand la fenêtre est passée.

In [5]:
r.delete("limite:malik")
for numero in range(1, 4):
    print("Malik, recherche", numero, ":", autoriser("malik", limite=2, fenetre=3))
time.sleep(3.5)
print("3,5 s plus tard :", autoriser("malik", limite=2, fenetre=3))

Malik, recherche 1 : True
Malik, recherche 2 : True
Malik, recherche 3 : False
3,5 s plus tard : True


## Exercice 7 : Une file de réservations

Une réservation demande du travail (vérifier les tables, prévenir le restaurant). Le site la
dépose dans une file, `file:reservations`, et répond aussitôt au client ; un programme à part, le
*worker*, la traite ensuite. On dépose à gauche (`LPUSH`), on retire à droite (`BRPOP`) : la
première arrivée est la première traitée.

**Question 1.** Écrivez `reserver(pseudo, id_resto, couverts, heure)` : elle range la
réservation (un dictionnaire, converti en JSON) en tête de la file et renvoie la longueur de la
file.

In [6]:
FILE = "file:reservations"

def reserver(pseudo, id_resto, couverts, heure):
    reservation = {"pseudo": pseudo, "restaurant": id_resto, "couverts": couverts, "heure": heure}
    # À COMPLÉTER : ranger la réservation, en JSON, en tête de la file
    return r.lpush(FILE, json.dumps(reservation))

**Question 2.** Déposez quatre réservations, puis regardez la file.

In [7]:
r.delete(FILE, "reservations:confirmees")   # on repart de files vides
reserver("lea", "41200846", 2, "19:30")
reserver("karim", "40732665", 4, "20:00")
reserver("sofia", "40383464", 2, "21:00")
reserver("malik", "50000923", 6, "19:00")
print(r.llen(FILE))
r.lrange(FILE, 0, -1)

4


['{"pseudo": "malik", "restaurant": "50000923", "couverts": 6, "heure": "19:00"}',
 '{"pseudo": "sofia", "restaurant": "40383464", "couverts": 2, "heure": "21:00"}',
 '{"pseudo": "karim", "restaurant": "40732665", "couverts": 4, "heure": "20:00"}',
 '{"pseudo": "lea", "restaurant": "41200846", "couverts": 2, "heure": "19:30"}']

**Question 3.** Écrivez le worker `traiter_file(delai=2)` : il retire une réservation avec
`r.brpop(FILE, timeout=delai)`, affiche une ligne, la range dans `reservations:confirmees`, et
recommence. `brpop` attend au plus `delai` secondes qu'un élément arrive ; s'il n'arrive rien, il
renvoie `None` et le worker s'arrête. Sinon il renvoie un couple `(nom de la file, élément)`.

In [8]:
def traiter_file(delai=2):
    nb_traitees = 0
    while True:
        # À COMPLÉTER : retirer un élément (brpop), s'arrêter s'il n'y en a pas
        element = r.brpop(FILE, timeout=delai)
        if element is None:
            break
        _, texte = element
        reservation = json.loads(texte)
        nom = r.hget(f"resto:{reservation['restaurant']}", "nom")
        print(f"Réservation traitée : {reservation['pseudo']}, {reservation['couverts']} couverts "
              f"chez {nom} à {reservation['heure']}")
        # À COMPLÉTER : ranger texte dans reservations:confirmees, compter
        r.lpush("reservations:confirmees", texte)
        nb_traitees += 1
    return nb_traitees

**Question 4.** Lancez le worker. Dans quel ordre les réservations sont-elles traitées ?
Combien de temps le worker attend-il avant de s'arrêter, et pourquoi ?

In [9]:
traiter_file()

Réservation traitée : lea, 2 couverts chez Lucali à 19:30
Réservation traitée : karim, 4 couverts chez Katz'S Delicatessen à 20:00
Réservation traitée : sofia, 2 couverts chez Le Bernardin à 21:00
Réservation traitée : malik, 6 couverts chez Di Fara Pizzeria à 19:00


4

**Question 5.** Que valent ensuite les longueurs des deux listes ?

In [10]:
print(r.llen(FILE), r.llen("reservations:confirmees"))

0 4


## Exercice 8 : Deux programmes, une file

Le worker tourne maintenant dans un autre programme : `worker_reservations.py`, dans le même
dossier. **Question 1** : lancez-le dans un deuxième terminal (commande dans le livret), puis
exécutez les cellules.

**Question 2.** Trois réservations arrivent, à une seconde d'intervalle. Regardez le terminal du
worker pendant que la cellule tourne. Quand chaque réservation est-elle traitée ? Que reste-t-il
dans la file ?

In [11]:
for pseudo, id_resto, couverts, heure in [("ines", "41461099", 3, "12:30"),
                                          ("lea", "40841272", 2, "13:00"),
                                          ("karim", "40364335", 5, "20:30")]:
    print("déposée, file de longueur", reserver(pseudo, id_resto, couverts, heure))
    time.sleep(1)
print("reste dans la file :", r.llen(FILE))

déposée, file de longueur 1
déposée, file de longueur 2
déposée, file de longueur 3
reste dans la file : 3


**Question 3.** Lancez un second worker dans un troisième terminal, puis déposez six réservations
d'un coup. Comment les deux workers se partagent-ils le travail ? Une réservation peut-elle être
traitée deux fois ?

In [12]:
for numero in range(1, 7):
    reserver("sofia", "41200846", numero + 1, "19:00")
time.sleep(3)
print("reste dans la file :", r.llen(FILE))
print("confirmées :", r.llen("reservations:confirmees"))

reste dans la file : 0
confirmées : 13


## Exercice 11 : Pour aller plus loin : une fenêtre glissante

**Question 1** (sur papier). Avec `autoriser`, combien de recherches un visiteur peut-il lancer
entre la 59e et la 61e seconde, s'il vise bien la fin de la fenêtre ?

La fenêtre glissante compte les requêtes des 60 dernières secondes, quelle que soit l'heure.
On range l'instant de chaque requête dans un ensemble trié (score = instant), on oublie les
plus anciennes, et on compte ce qui reste.

**Question 2.** Complétez `autoriser_glissant`.

In [13]:
def autoriser_glissant(pseudo, limite=5, fenetre=60):
    cle = f"limite_glissante:{pseudo}"
    maintenant = time.time()
    # À COMPLÉTER
    r.zremrangebyscore(cle, 0, maintenant - fenetre)   # oublier les requêtes plus vieilles que fenetre secondes
    if r.zcard(cle) >= limite:                          # refuser si l'ensemble contient déjà limite requêtes
        return False
    r.zadd(cle, {str(maintenant): maintenant})          # noter l'instant de cette requête
    r.expire(cle, fenetre)                              # donner à la clé une durée de vie de fenetre secondes
    return True

**Question 3.** Testez avec une limite de 2 requêtes par fenêtre de 3 secondes. Expliquez les
trois lignes affichées.

In [14]:
r.delete("limite_glissante:ines")
print("t = 0 s :", autoriser_glissant("ines", 2, 3), autoriser_glissant("ines", 2, 3), autoriser_glissant("ines", 2, 3))
time.sleep(2)
print("t = 2 s :", autoriser_glissant("ines", 2, 3))
time.sleep(1.5)
print("t = 3,5 s :", autoriser_glissant("ines", 2, 3), autoriser_glissant("ines", 2, 3), autoriser_glissant("ines", 2, 3))

t = 0 s : True True False
t = 2 s : False
t = 3,5 s : True True False
